# Phase 1f — Qualitative & Financial Analysis

This dedicated analysis gate serves as the **early safety validation** for Phase 1 (Model Selection & Reasoning Effort). It compiles the absolute business valuation metrics and the GPT-5.4 explainability fingerprints for all candidate models and reasoning levels before any prompt sweeps or hybrid blend tunings are executed.

## Portfolio Simulation Controls
Adjust loan volume and risk parameters here. All ledger calculations below update dynamically.

In [1]:
# ==========================================
# BANC SABADELL PORTFOLIO SIMULATION CONTROLS
# ==========================================
PORTFOLIO_SIZE = 1000          # Number of credit applications to score
DEFAULT_RATE = 0.15           # Expected default rate (15.0%)
AVERAGE_LOAN_AMOUNT = 10000    # Average loan size in USD/EUR
LGD = 0.50                    # Expected Loss Given Default (50% lost on default)
EXPECTED_PROFIT = 2000         # Expected interest profit per repaid loan (USD/EUR)
# ==========================================
print(f"Simulation parameters set: Portfolio={PORTFOLIO_SIZE}, Default Rate={DEFAULT_RATE*100}%, Loan Size={AVERAGE_LOAN_AMOUNT}")

Simulation parameters set: Portfolio=1000, Default Rate=15.0%, Loan Size=10000


In [2]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from llm_utils import call_llm, load_api_key

DATA_DIR = "../../../data"
RESULTS_DIR = "../../../data/results"
LLM_CALLS_PATH = "../../../data/results/llm/llm_calls.csv"
PREDS_1A_PATH = "../../../data/results/llm/01a_predictions.csv"
PREDS_1D_PATH = "../../../data/results/llm/01d_predictions.csv"

# Load OpenAI API keys from .env for key-rotation and parallel execution
load_api_key("openai")
_key1 = os.environ.get("OPENAI_API_KEY")
_key2 = os.environ.get("OPENAI_API_KEY_2") or _key1
_key3 = os.environ.get("OPENAI_API_KEY_3") or _key1

API_KEYS = [_key1, _key2, _key3]
API_KEYS = [k for k in API_KEYS if k]
assert API_KEYS, "Set OPENAI_API_KEY in notebooks/llm_models/.env"

def run_financial_simulation(metrics_df_path, phase, baseline_condition_name=None):
    metrics_df = pd.read_csv(metrics_df_path)
    
    if os.path.exists(LLM_CALLS_PATH):
        calls = pd.read_csv(LLM_CALLS_PATH)
    else:
        calls = pd.DataFrame()
        
    defaults = PORTFOLIO_SIZE * DEFAULT_RATE
    good_loans = PORTFOLIO_SIZE * (1.0 - DEFAULT_RATE)
    lgd_cost = AVERAGE_LOAN_AMOUNT * LGD
    
    results = []
    for _, row in metrics_df.iterrows():
        if 'model' in row and 'condition' in row:
            condition = f"{row['model']} ({row['condition']})"
        elif 'condition' in row:
            condition = str(row['condition'])
        elif 'variant' in row:
            condition = str(row['variant'])
        else:
            condition = str(row.iloc[0])
            
        recall = row.get('co_recall_mean', row.get('recall_charged_off', np.nan))
        if pd.isna(recall):
            recall = row.get('recall', 0.0)
            
        precision = row.get('co_precision_mean', row.get('precision_charged_off', np.nan))
        if pd.isna(precision):
            precision = row.get('precision', 0.0)
            
        if 'XGBoost' in condition or 'structured' in condition:
            cost_100 = 0.0
        elif 'cost_mean' in row:
            cost_100 = row['cost_mean']
        else:
            cost_100 = 0.0
            if not calls.empty:
                if phase == '1a':
                    model_name = row['model']
                    cond_name = row['condition']
                    sub = calls[(calls['notebook_id'] == '01a_Model_Comparison.ipynb') & 
                                (calls['desc_tag'] == cond_name) &
                                (calls['label'].str.contains(model_name, case=False, na=False))]
                    if not sub.empty:
                        cached_costs = []
                        for _, call_row in sub.iterrows():
                            in_t = call_row.get('input_tokens', 0)
                            out_t = call_row.get('output_tokens', 0)
                            p_in = call_row.get('input_price_per_1k_usd', 0)
                            p_out = call_row.get('output_price_per_1k_usd', 0)
                            
                            c = 200 if in_t > 1024 else 0
                            if in_t > 1024 and c > 0:
                                cached_t = min(in_t, c)
                                uncached_t = in_t - cached_t
                                cost = ( (uncached_t * p_in) + (cached_t * (p_in * 0.5)) + (out_t * p_out) ) / 1000.0
                            else:
                                cost = ( (in_t * p_in) + (out_t * p_out) ) / 1000.0
                            cached_costs.append(cost)
                        cost_100 = 100.0 * sum(cached_costs) / len(cached_costs)
                elif phase == '1d':
                    effort = row['reasoning_effort']
                    sub = calls[(calls['notebook_id'] == '01d_reasoning_effort_runs.ipynb') & 
                                (calls['label'] == f'GPT-5.4 reasoning={effort}')]
                    if not sub.empty:
                        cached_costs = []
                        for _, call_row in sub.iterrows():
                            in_t = call_row.get('input_tokens', 0)
                            out_t = call_row.get('output_tokens', 0)
                            p_in = call_row.get('input_price_per_1k_usd', 0)
                            p_out = call_row.get('output_price_per_1k_usd', 0)
                            
                            c = 200 if in_t > 1024 else 0
                            if in_t > 1024 and c > 0:
                                cached_t = min(in_t, c)
                                uncached_t = in_t - cached_t
                                cost = ( (uncached_t * p_in) + (cached_t * (p_in * 0.5)) + (out_t * p_out) ) / 1000.0
                            else:
                                cost = ( (in_t * p_in) + (out_t * p_out) ) / 1000.0
                            cached_costs.append(cost)
                        cost_100 = 100.0 * sum(cached_costs) / len(cached_costs)
        
        tp = defaults * recall
        fn = defaults - tp
        total_rejections = tp / precision if precision > 0 else 0
        fp = total_rejections - tp
        tn = good_loans - fp
        
        loss_from_missed_defaults = fn * lgd_cost
        lost_profit_from_false_rejections = fp * EXPECTED_PROFIT
        api_cost = (cost_100 / 100.0) * PORTFOLIO_SIZE
        total_bank_cost = loss_from_missed_defaults + lost_profit_from_false_rejections + api_cost
        
        results.append({
            "Condition": condition,
            "Defaults Caught (TP)": f"{tp:.1f} ({recall*100:.1f}%)",
            "False Rejections (FP)": f"{fp:.1f}",
            "Credit Default Loss": loss_from_missed_defaults,
            "False Rejections Lost Profit": lost_profit_from_false_rejections,
            "API Token Cost": api_cost,
            "Total Cost": total_bank_cost
        })
        
    sim_df = pd.DataFrame(results).set_index("Condition")
    
    if baseline_condition_name and baseline_condition_name in sim_df.index:
        control_total = sim_df.loc[baseline_condition_name, "Total Cost"]
        sim_df["Net Financial Impact ($)"] = sim_df["Total Cost"] - control_total
        if control_total > 0:
            sim_df["Net Financial Impact (%)"] = (sim_df["Net Financial Impact ($)"] / control_total) * 100.0
        else:
            sim_df["Net Financial Impact (%)"] = 0.0
    else:
        sim_df["Net Financial Impact ($)"] = 0.0
        sim_df["Net Financial Impact (%)"] = 0.0
        
    formatted_df = sim_df.copy()
    formatted_df["Credit Default Loss"] = formatted_df["Credit Default Loss"].map("${:,.2f}".format)
    formatted_df["False Rejections Lost Profit"] = formatted_df["False Rejections Lost Profit"].map("${:,.2f}".format)
    formatted_df["API Token Cost"] = formatted_df["API Token Cost"].map("${:,.2f}".format)
    formatted_df["Total Cost"] = formatted_df["Total Cost"].map("${:,.2f}".format)
    formatted_df["Net Financial Impact ($)"] = formatted_df["Net Financial Impact ($)"].map(
        lambda x: f"+${x:,.2f}" if x > 0 else (f"-${abs(x):,.2f}" if x < 0 else "$0.00")
    )
    formatted_df["Net Financial Impact (%)"] = formatted_df["Net Financial Impact (%)"].map(
        lambda x: f"+{x:.2f}%" if x > 0 else (f"-{abs(x):.2f}%" if x < 0 else "0.00%")
    )
    return formatted_df

print("Cost simulation engine initialized.")

Cost simulation engine initialized.


## Part 1 — Credit Economics Ledgers

In [3]:
print("=== Phase 1a: Model Selection Financial Ledger ===")
phase1a_df = run_financial_simulation(f"{RESULTS_DIR}/llm/01a_metrics.csv", '1a', "XGBoost (structured)")
display(phase1a_df)

print("\n=== Phase 1d: Reasoning Effort Financial Ledger ===")
phase1d_df = run_financial_simulation(f"{RESULTS_DIR}/llm/01d_metrics.csv", '1d', "XGBoost (tuning)")
display(phase1d_df)

=== Phase 1a: Model Selection Financial Ledger ===


,Defaults Caught (TP),False Rejections (FP),Credit Default Loss,False Rejections Lost Profit,API Token Cost,Total Cost,Net Financial Impact ($),Net Financial Impact (%)
Condition,,,,,,,,
XGBoost (structured),70.0 (46.7%),270.0,"$400,000.00","$540,000.00",$0.00,"$940,000.00",$0.00,0.00%
GPT-5.4 (no_desc),60.0 (40.0%),100.0,"$450,000.00","$200,000.00",$2.51,"$650,002.51","-$289,997.49",-30.85%
GPT-5.4 (with_desc),60.0 (40.0%),130.0,"$450,000.00","$260,000.00",$2.66,"$710,002.66","-$229,997.34",-24.47%
Claude Opus 4.8 (no_desc),70.0 (46.7%),250.0,"$400,000.00","$500,000.00",$6.16,"$900,006.16","-$39,993.84",-4.25%
Claude Opus 4.8 (with_desc),70.0 (46.7%),230.0,"$400,000.00","$460,000.00",$6.70,"$860,006.70","-$79,993.30",-8.51%
Claude Sonnet 4.6 (with_desc),90.0 (60.0%),440.0,"$300,000.00","$880,000.00",$3.81,"$1,180,003.81","+$240,003.81",+25.53%
Gemini 3.5 Flash (no_desc),80.0 (53.3%),300.0,"$350,000.00","$600,000.00",$1.51,"$950,001.51","+$10,001.51",+1.06%
Gemini 3.5 Flash (with_desc),80.0 (53.3%),290.0,"$350,000.00","$580,000.00",$1.60,"$930,001.60","-$9,998.40",-1.06%
Gemini 2.5 Pro (no_desc),110.0 (73.3%),410.0,"$200,000.00","$820,000.00",$1.40,"$1,020,001.40","+$80,001.40",+8.51%



=== Phase 1d: Reasoning Effort Financial Ledger ===


,Defaults Caught (TP),False Rejections (FP),Credit Default Loss,False Rejections Lost Profit,API Token Cost,Total Cost,Net Financial Impact ($),Net Financial Impact (%)
Condition,,,,,,,,
GPT-5.4 reasoning=low,70.0 (46.7%),180.0,"$400,000.00","$360,000.00",$3.02,"$760,003.02","-$179,996.98",-19.15%
GPT-5.4 reasoning=medium,50.0 (33.3%),140.0,"$500,000.00","$280,000.00",$4.82,"$780,004.82","-$159,995.18",-17.02%
GPT-5.4 reasoning=high,60.0 (40.0%),110.0,"$450,000.00","$220,000.00",$6.58,"$670,006.58","-$269,993.42",-28.72%
XGBoost (tuning),70.0 (46.7%),270.0,"$400,000.00","$540,000.00",$0.00,"$940,000.00",$0.00,0.00%


## Part 2 — Qualitative Reasoning Fingerprinting

We sample 10 correct/incorrect reasoning chains for each candidate model and reasoning effort, then run the GPT-5.4 qualitative judge to identify feature anchors, speculation bias, or explainability flaws.

In [4]:
import random
import json

JUDGE_SYSTEM = (
    "You are reviewing how an AI credit risk model reasons about loan applications. "
    "Write a concise qualitative characterisation (4-6 sentences) of what is DISTINCTIVE "
    "about this model's decision reasoning style. Describe:\n"
    "- Which loan features or signals it consistently anchors on\n"
    "- Its overall risk posture (e.g. conservative, optimistic, balanced, formulaic)\n"
    "- Any systematic patterns, blind spots, or tendencies across samples\n"
    "- Whether its reasoning feels specific to each loan or generic and templated\n\n"
    "Do NOT score or rank. Do NOT say whether the model is good or bad. "
    "Just describe its reasoning fingerprint as you would describe a person's decision-making style."
)

qual_results = {}
out_path = f"{RESULTS_DIR}/llm/01f_qualitative_financial.json"
FORCE_RERUN_JUDGE = False  # Set to True to re-run all judge LLM calls

if not FORCE_RERUN_JUDGE and os.path.exists(out_path):
    print(f"Found existing qualitative analysis results at: {out_path}! Loading instead of calling OpenAI API...")
    with open(out_path, "r", encoding="utf-8") as f:
        qual_results = json.load(f)

def run_one_judge_task(idx, tgt):
    df = pd.read_csv(tgt["file"])
    sub = df[df[tgt["filter_col"]].astype(str).str.contains(tgt["filter_val"], case=False, na=False)]
    if tgt["desc_col"] is not None and tgt["desc_val"] is not None:
        sub = sub[sub[tgt["desc_col"]] == tgt["desc_val"]]
    if sub.empty:
        return tgt["key"], "No predictions or reasoning logs found in metrics CSV."
    if "llm_correct" in sub.columns:
        correct_df = sub[sub["llm_correct"] == 1]
        incorrect_df = sub[sub["llm_correct"] == 0]
    else:
        correct_df = sub[sub["actual"] == sub["llm_pred"]]
        incorrect_df = sub[sub["actual"] != sub["llm_pred"]]
    chosen_correct = correct_df.sample(min(len(correct_df), 5), random_state=42) if len(correct_df) > 0 else pd.DataFrame()
    chosen_incorrect = incorrect_df.sample(min(len(incorrect_df), 5), random_state=42) if len(incorrect_df) > 0 else pd.DataFrame()
    chosen = pd.concat([chosen_correct, chosen_incorrect])
    lines = []
    for r_idx, (_, r_row) in enumerate(chosen.iterrows(), 1):
        act_lbl = "Fully Paid" if r_row['actual'] == 1 else "Charged Off"
        pred_lbl = "Fully Paid" if r_row['llm_pred'] == 1 else "Charged Off"
        status = "correct" if r_row['actual'] == r_row['llm_pred'] else "WRONG"
        reasons_text = r_row.get('llm_reasoning', '')
        lines.append(f"[{r_idx}] Actual: {act_lbl} | Predicted: {pred_lbl} ({status})\n"
                     f"    Reasoning: {reasons_text}")
    user_prompt = (
        f"Model Variant: {tgt['key']}\n"
        f"--- REASONING SAMPLES ---\n"
        + "\n\n".join(lines)
        + "\n--- END SAMPLES ---"
    )
    key_val = API_KEYS[idx % len(API_KEYS)]
    print(f"Judging {tgt['key']} on key {idx % len(API_KEYS) + 1}...")
    characterisation = call_llm(
        JUDGE_SYSTEM, user_prompt, 
        api_provider="openai", model="gpt-5.4", api_key=key_val
    )
    return tgt["key"], characterisation

# Identify missing targets
missing_targets = [t for t in TARGETS if t["key"] not in qual_results]

if missing_targets:
    print(f"Running LLM judge for {len(missing_targets)} missing targets in parallel...")
    with ThreadPoolExecutor(max_workers=min(len(API_KEYS), 4)) as ex:
        futures = {ex.submit(run_one_judge_task, i, t): t["key"] for i, t in enumerate(missing_targets)}
        for fut in as_completed(futures):
            k, v = fut.result()
            qual_results[k] = v
            print(f"Finished judging: {k}")

    # Save qualitative findings to JSON
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(qual_results, f, indent=2, ensure_ascii=False)
    print(f"\nSaved qualitative + financial datasets to: {out_path}")
else:
    print("\nAll targets are already qualitatively characterized and loaded!")


Judging GPT-5.4 (with_desc) on key 2...Judging GPT-5.4 (no_desc) on key 1...
Judging Claude Opus 4.8 (no_desc) on key 3...



Finished judging: Claude Opus 4.8 (no_desc)
Judging Claude Opus 4.8 (with_desc) on key 1...
Finished judging: GPT-5.4 (with_desc)
Judging Claude Sonnet 4.6 (with_desc) on key 2...


Finished judging: GPT-5.4 (no_desc)
Judging Gemini 3.5 Flash (no_desc) on key 3...


Finished judging: Claude Opus 4.8 (with_desc)
Judging Gemini 3.5 Flash (with_desc) on key 1...


Finished judging: Claude Sonnet 4.6 (with_desc)
Judging Gemini 2.5 Pro (no_desc) on key 2...
Finished judging: Gemini 3.5 Flash (no_desc)
Judging Gemini 2.5 Pro (with_desc) on key 3...


Finished judging: Gemini 3.5 Flash (with_desc)
Judging GPT-5.4 reasoning=low on key 1...
Finished judging: Gemini 2.5 Pro (no_desc)
Judging GPT-5.4 reasoning=medium on key 2...
Finished judging: Gemini 2.5 Pro (with_desc)
Judging GPT-5.4 reasoning=high on key 3...


Finished judging: GPT-5.4 reasoning=medium


Finished judging: GPT-5.4 reasoning=high


Finished judging: GPT-5.4 reasoning=low

Saved qualitative + financial datasets to: ../../../data/results/llm/01f_qualitative_financial.json


In [ ]:
# ── Pretty-print qualitative fingerprints ─────────────────────────────────
from IPython.display import display, Markdown

if qual_results:
    display(Markdown("---\n## Qualitative Reasoning Fingerprints"))
    for key, fingerprint in qual_results.items():
        formatted_fingerprint = fingerprint.replace('\n', '  \n> ')
        display(Markdown(
            f"### {key}\n"
            f"> {formatted_fingerprint}"
        ))
else:
    print("No qualitative results yet — run the judge cell first.")
